In [0]:
%sql
MERGE INTO capgeminipro.retail_gold.dim_customers tgt

USING (

    SELECT
        CustomerID,
        CustomerName,
        Email,
        City,
        Address,
        LastUpdated

    FROM capgeminipro.retail_silver.silver_customers_delta

) src

ON tgt.CustomerID = src.CustomerID
AND tgt.IsCurrent = 'Y'

WHEN MATCHED AND (

       tgt.City <> src.City
    OR tgt.Address <> src.Address
    OR tgt.CustomerName <> src.CustomerName

)

THEN UPDATE SET

    tgt.EndDate = current_date() - 1,
    tgt.IsCurrent = 'N'

WHEN NOT MATCHED

THEN INSERT (

    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsCurrent

)

VALUES (

    src.CustomerID,
    src.CustomerName,
    src.Email,
    src.City,
    src.Address,
    src.LastUpdated,
    NULL,
    'Y'

);